# Elevation

Topography vs street network (mosaic) structure

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import matplotlib as mpl
# standardized parameters for thesis figures
mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})

cm = 1 / 2.54

In [ ]:
import utca
import osmnx as ox
from pathlib import Path
import pandas as pd

In [ ]:
# paths to saved simplified graphs and elevation data files
folder = Path('output/neat_20260103_171612')
elevation_path = [f'data/GEO/eu_dem_v11_E{x}0N20.tif' for x in [4,5]]

### Distribution of node types by elevation

In [ ]:
# HUSL palette (distinct categorical colors)
palette = sns.color_palette("hls", 3)
#palette = sns.mpl_palette('tab10', 3)
# Add a neutral color for the overflow bin (15+)
palette.append((0.5, 0.5, 0.5))  # dark gray
cmap = mcolors.ListedColormap(palette)

In [ ]:
palette

In [ ]:
def plot_boxplot(cityname="Pécs", save=False):
    G = utca.process_town_elev(cityname, folder, elevation_path)
    node_data = ox.graph_to_gdfs(G, edges=False)
    fig, ax = plt.subplots(figsize=(8 * cm, 7 * cm))
    sns.boxplot(
        x="type",
        y="max_grade",
        #order=["X", "Y", "T", "other"],
        hue="type",
        palette=palette,
        saturation=1,
        data=node_data,
        ax=ax,
    )
    ax.set_xlabel("Type")
    ax.set_ylabel("Max grade")
    # plt.title('Max grade by Node Type')
    plt.show()
    if save:
        fig.savefig("output/figs_jan2/elev_boxplot.pdf")

In [ ]:
plot_boxplot(cityname="Pécs", save=False)

### Map plotting with elevation/street grade data

In [ ]:
def plot_elev_map(cityname, save=False):
    fig, axs = plt.subplots(1, 2, figsize=(14 * cm, 8 * cm))
    G = utca.process_town_elev(cityname, folder, elevation_path)
    nodes, edges = ox.graph_to_gdfs(G)
    nodes.plot(
        ax=axs[0],
        column="type",
        legend=True,
        cmap=cmap,
        markersize=0.1,
        categories=["Y", "X", "T", "other"],
        legend_kwds={"title": "Type"},
    )
    edges.plot(ax=axs[0], color="black", alpha=0.5, linewidth=0.2)
    axs[0].axis("off")
    edges.plot(
        ax=axs[1],
        column="grade_abs",
        legend=True,
        cmap="plasma",
        linewidth=0.5,
        legend_kwds={"label": "Grade", "shrink": 0.7},
    )
    axs[1].axis("off")
    if save:
        fig.savefig("output/figs_jan2/elev_map.png", dpi=300)

In [ ]:
plot_elev_map("Pécs")

## Town scale

Characterizing towns by elevation (how hilly or how flat they are)

Read data: can be generated with `calculate_elevation_stdev.py`.

In [ ]:
elev_stdevs = pd.read_csv("output/elev_stdevs_jan3.csv")

Read graph stats data: can be generated with ???

In [ ]:
all_graph_stats = pd.read_csv("output/all_graph_stats_jan3.csv")
all_graph_stats.fillna(0, inplace=True)
all_graph_stats['XYT'] = all_graph_stats['X'] + all_graph_stats['Y'] + all_graph_stats['T']
all_graph_stats['X^'] = all_graph_stats['X'] / all_graph_stats['XYT']
all_graph_stats['Y^'] = all_graph_stats['Y'] / all_graph_stats['XYT']
all_graph_stats['T^'] = all_graph_stats['T'] / all_graph_stats['XYT']

In [ ]:
joined = pd.merge(all_graph_stats, elev_stdevs, 'left', 'cityname')

Population stats for filtering, can be generated with ???

In [ ]:
pop_stats = pd.read_csv("output/stats_dec16.csv")

In [ ]:
joined = pd.merge(joined, pop_stats[['cityname', 'nepesseg']], 'left', on='cityname')

Use only towns with more than 10k population

In [ ]:
tizezer = joined[joined['nepesseg'] > 10000]

### Plot town elevation stdev data on symbolic plane

In [ ]:
def plot_all_scatter(joined, save=False):
    fig, ax = plt.subplots(figsize=(9*cm, 6*cm))
    #ax.scatter(x='n', y='v', c='elev_std', s=0.5, data=joined, cmap='plasma')
    joined.plot.scatter(ax=ax, x='n', y='v', c='elev_std', s=2, cmap='plasma')#, alpha=0.8)
    #ax.set_ylim(top=5)
    ax.set_xlabel("$\\overline{n}$")
    ax.set_ylabel("$\\overline{v}$  ")#, rotation=0)
    if save:
        fig.savefig("output/figs_jan2/elev_scatter.pdf")

In [ ]:
plot_all_scatter(tizezer, False)

### Calculate elevation stdev correlations

In [ ]:
tizezer[['elev_std', 'F', 'V',
       'N_F', 'N_V', 'v', 'n', 'X^', 'Y^', 'T^',]].corr()

In [ ]:
len(tizezer)

In [ ]:
corr = tizezer[['elev_std', 'F', 'V',
       'N_F', 'N_V', 'v', 'n', 'X^', 'Y^', 'T^',]].corr()

LaTeX table for thesis:

In [ ]:
print(corr['elev_std'].to_latex(float_format="%.4f"))